# K-matrix validation

Laura++ / Anisovich–Sarantsev five-pole, five-channel $\pi\pi$ S-wave. This notebook validates the phase-space channels, the physical production amplitude, coupled-channel unitarity, the Dalitz density, and a toy MC.

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dalitzplotfitter import DecayChannel, DecayModel, KMatrix, RealImag, Resonance, enable_x64, weighted_resample

enable_x64()

In [ ]:
channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))
mpi_minus, mpi_plus, _ = channel.daughter_masses

kmatrix = KMatrix(
    betas=(
        RealImag(1.00, 0.00),
        RealImag(0.35, -0.20),
        RealImag(-0.15, 0.25),
        RealImag(0.10, 0.05),
        RealImag(0.05, -0.08),
    ),
    f_prod=(
        RealImag(0.20, 0.10),
        RealImag(-0.08, 0.04),
        RealImag(0.03, -0.02),
        RealImag(0.00, 0.00),
        RealImag(0.00, 0.00),
    ),
)

m = jnp.linspace(2.0 * mpi_plus + 1e-4, 1.72, 2200)
labels = [r"$\pi\pi$", r"$K\bar K$", r"$4\pi$", r"$\eta\eta$", r"$\eta\eta'$"]

## Phase space and $K(s)$

In [ ]:
rho = np.asarray(kmatrix.phase_space(m))
K = np.asarray(kmatrix.scattering_matrix(m))
print("max symmetry residual =", np.max(np.abs(K - np.swapaxes(K, -1, -2))))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), constrained_layout=True)
for i, label in enumerate(labels):
    axes[0].plot(np.asarray(m), rho[:, i].real, label=label)
    axes[1].plot(np.asarray(m), rho[:, i].imag, label=label)
axes[0].set(xlabel=r"$m_{\pi\pi}$ [GeV]", ylabel=r"Re $\rho_i$")
axes[1].set(xlabel=r"$m_{\pi\pi}$ [GeV]", ylabel=r"Im $\rho_i$")
axes[0].legend(ncol=2); axes[1].legend(ncol=2)
plt.show()

fig, ax = plt.subplots(figsize=(9, 5.5))
for j, label in enumerate(labels):
    values = K[:, 0, j]
    finite = np.isfinite(values)
    ax.plot(np.asarray(m)[finite], values[finite], label=f"K_1{j+1} {label}")
ax.set(xlabel=r"$m_{\pi\pi}$ [GeV]", ylabel=r"$K_{1j}(s)$", ylim=(-20, 20))
ax.legend(ncol=2)
plt.show()

## Coupled-channel unitarity

For $T=(I-iK\rho)^{-1}K$ define $S=I+2i\sqrt{\rho}T\sqrt{\rho}$. Above the highest included threshold all five channels are open and the full matrix must satisfy $S^\dagger S=I$.

In [ ]:
m_all_open_threshold = 0.547862 + 0.95778
m_u = jnp.linspace(m_all_open_threshold + 1e-4, 1.72, 900)
S = np.asarray(kmatrix.s_matrix(m_u))
SdagS = np.swapaxes(np.conj(S), -1, -2) @ S
residual = SdagS - np.eye(5)[None, :, :]
frob = np.linalg.norm(residual, axis=(-2, -1))
maxel = np.max(np.abs(residual), axis=(-2, -1))
eigvals = np.linalg.eigvalsh(0.5 * (SdagS + np.swapaxes(np.conj(SdagS), -1, -2)))

print("all-open threshold =", m_all_open_threshold, "GeV")
print("max Frobenius residual =", np.max(frob))
print("max elementwise residual =", np.max(maxel))
print("max |eigenvalue - 1| =", np.max(np.abs(eigvals - 1.0)))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), constrained_layout=True)
axes[0].semilogy(np.asarray(m_u), frob, label=r"$||S^\dagger S-I||_F$")
axes[0].semilogy(np.asarray(m_u), maxel, label=r"$\max |(S^\dagger S-I)_{ij}|$")
axes[0].set(xlabel=r"$m_{\pi\pi}$ [GeV]", ylabel="unitarity residual")
axes[0].legend()
for i in range(5):
    axes[1].plot(np.asarray(m_u), eigvals[:, i], label=f"eigenvalue {i+1}")
axes[1].axhline(1.0, linestyle="--", linewidth=1.0)
axes[1].set(xlabel=r"$m_{\pi\pi}$ [GeV]", ylabel=r"eigenvalues of $S^\dagger S$")
axes[1].legend(ncol=2)
plt.show()

T = np.asarray(kmatrix.scattering_amplitude(m_u))
rho_u = np.asarray(kmatrix.phase_space(m_u))
t11 = rho_u[:, 0] * T[:, 0, 0]
fig, ax = plt.subplots(figsize=(6.5, 6.0))
ax.plot(t11.real, t11.imag)
ax.set(xlabel=r"Re $[\sqrt{\rho}T\sqrt{\rho}]_{11}$", ylabel=r"Im $[\sqrt{\rho}T\sqrt{\rho}]_{11}$")
ax.set_aspect("equal", adjustable="datalim")
plt.show()

## P-vector and physical $\pi\pi$ S-wave

In [ ]:
P = np.asarray(kmatrix.production_vector(m))
F = np.asarray(kmatrix.amplitude_vector(m))
F1 = F[:, 0]
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
axes[0,0].plot(np.asarray(m), F1.real); axes[0,0].set_ylabel(r"Re $F_1$")
axes[0,1].plot(np.asarray(m), F1.imag); axes[0,1].set_ylabel(r"Im $F_1$")
axes[1,0].plot(np.asarray(m), np.abs(F1)**2); axes[1,0].set_ylabel(r"$|F_1|^2$")
axes[1,1].plot(np.asarray(m), np.unwrap(np.angle(F1))); axes[1,1].set_ylabel("phase [rad]")
for ax in axes.flat: ax.set_xlabel(r"$m_{\pi\pi}$ [GeV]")
plt.show()

## Dalitz density and toy MC

In [ ]:
model = DecayModel(
    channel,
    [Resonance("pipi_S_kmatrix", pair=(0,1), coefficient=RealImag(1.0,0.0), mass=1.0, width=0.0, spin=0, lineshape=kmatrix, resonance_radius=3.0, parent_radius=3.0)],
    normalization_resolution=700,
    normalization_boundary_resolution=20001,
)
grid = model.normalization_sample
intensity = np.asarray(model.intensity(grid.as_dict()))
weights = np.asarray(grid.weights) * intensity
fig, ax = plt.subplots(figsize=(7,6))
h = ax.hist2d(np.asarray(grid.s12), np.asarray(grid.s13), bins=120, weights=weights)
fig.colorbar(h[3], ax=ax)
ax.set(xlabel=r"$s_{12}$ [GeV$^2$]", ylabel=r"$s_{13}$ [GeV$^2$]")
plt.show()

N_POOL, N_TOY = 1_000_000, 100_000
pool = model.generate_phase_space(N_POOL, seed=4100)
target_weights = pool.weights * model.intensity(pool.as_dict())
toy = weighted_resample(jax.random.key(4101), pool, target_weights, N_TOY, replace=True)
fig, ax = plt.subplots(figsize=(7,6))
h = ax.hist2d(np.asarray(toy.s12), np.asarray(toy.s13), bins=100)
fig.colorbar(h[3], ax=ax)
ax.set(xlabel=r"$s_{12}$ [GeV$^2$]", ylabel=r"$s_{13}$ [GeV$^2$]")
plt.show()